<div style="display: flex; align-items: center; justify-content: center; gap: 20px;">
  <!-- Left Column: Image -->
  <div style="flex: 1; text-align: center;">
    <img src="https://i0.wp.com/cdcs.web.ua.pt/wp-content/uploads/2022/05/cropped-cropped-Picture13-1.png?w=968" width="370" height="200" style="display: block; margin: auto;"/>
  </div>

  <!-- Right Column: Text -->
  <div style="flex: 1; text-align: left;">
    <div><strong style="color: #4F5B63; font-size: 1.5em;">Master in Data Science for Social Sciences</strong></div>
    <div><strong style="color: #4F5B63; font-size: 1.2em;">University of Aveiro</strong></div>
    <p style="color: #46627F; font-weight: bold; font-style: italic;">Introduction to Data Science - 2026/2027</p>
    <p style="color: #4F5B63;">João Lourenço Marques</p>
    <p style="color: #4F5B63;">Paulo Batista</p>
  </div>
</div>

<div style="display: flex; justify-content: space-around; align-items: flex-start;">
  <div style="width: 100%; padding: 10px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); margin: 10px;">
    <h2><h1 style="text-align: center; font-size: 4em; color: #46627F; margin-top: 0; margin-bottom: 0; line-height: 1;">Natural Language Processing (and Text Mining)</h1>
    <p>
<h1 style="text-align: center; color: #B1C0CF; margin-top: 0; margin-bottom: 0; line-height: 1;"> - Exploratory Natural Language Processing based on GOVCOPP data - </h1></p></h2>
      </div>
</div>

# SETUP

In [ ]:
# Setup — do not edit. Finds the project folder, its data and its outputs.
from pathlib import Path

def _project_root(marker="environment.yml"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / marker).exists():
            return folder
    raise FileNotFoundError(f"No {marker} in {here} or above: open this notebook inside the project folder")

PROJECT_ROOT = _project_root()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Turn a corpus into features and inspect lexical similarity

**Goal of the practice block.** Starting from the GOVCOPP publications table, declare the corpus
and its textual unit, rebuild the course's TF-IDF representation of the abstracts, read two fixed
pairs of documents against their original text, and run one preprocessing contrast that changes a
single decision.

**Project question.** To what extent does the formal organisational structure of the GOVCOPP
research groups correspond to the thematic structure observable in their researchers'
scientific production?

**Today's question.** Which textual information do our features retain, and when does lexical
proximity fail to support a thematic interpretation?

W06 described *who signs with whom*. Today we start reading *what the publications say*, through
their abstracts. The publication stays the unit: no researcher or group profile is built today
(that comes in W08–W09), and no topic model either.

**Learning outcomes:** OA5.1, OA5.2, OA9.1.

| Section | When | Minutes | Evidence it produces |
|---|---|---:|---|
| 1–3 Fictitious corpus | exposition and demo | 30–55, 75–90 | counts, TF-IDF and cosine by hand, confirmed by code |
| 4 Corpus and textual unit | groups | 100–125 | declared corpus, field rule and language check |
| 5–6 The reference representation (checkpoint C1) | groups | 100–125 | vocabulary and matrix-shape check |
| 7 Two fixed pairs (C2) | groups | 125–150 | two neighbour examples read against the abstracts |
| 8 One contrast, one factor (C3) | groups | 150–180 | one changed setting, read in real documents |
| 9 Decision log, evidence record, outputs | groups | 150–180 | updated evidence record with parameters and provenance |
| 10 Optional extension | if you finish | — | the fixed pairs under your contrast |
| After class | home extension | about 4 h per student | second pair if unfinished, entry E07, Research Plan v1, M2 dossier |

**The minimum in class.** Checkpoint C1, the first fixed pair (C2) and your contrast run and
recorded (C3). The second pair and the written justification of the contrast may be finished at
home.

**How to work.** Groups of three or four, with rotating roles: *operator* (runs the notebook),
*data checker* (compares every result with the checkpoint), *reader* (opens the original abstracts
and checks every interpretation against them). Run the cells in order. Run each checkpoint before
moving on: if it fails, fix the cell above it. Finish with **Kernel → Restart & Run All**.

**Packages.** pandas, numpy, matplotlib and scikit-learn, all already in the W05 `icd-project`
environment. Nothing to install today.

| You may change | You must check | You must keep |
|---|---|---|
| the cells marked `YOUR CODE HERE` | every `assert` checkpoint | the file name in `data/raw/` |
| the Markdown answer cells | the unit of each table | the parameters cell, except `MY_CONTRAST` |
| `MY_CONTRAST`, once, in section 8 | Restart & Run All at the end | the order of the sections |

**Where this notebook lives: the W05 project rules.** This is the second notebook of the GOVCOPP
project. It runs inside your group's project folder, next to the W06 notebook.

| W05 rule | What it means in this notebook |
|---|---|
| The project, not the notebook, is the unit of work | Save it as `notebooks/W07S01_practice.ipynb` in your group's repository |
| Build paths from the project root | The first cell finds the folder that holds `environment.yml`; never type `C:/…` |
| Raw data are read, never edited | The publications CSV stays in `data/raw/` as received; the fingerprint check in section 4 proves it |
| Outputs are regenerated, never retouched | Run All rewrites `outputs/text_representation/`; never edit a CSV or a figure by hand |
| Restart Kernel → Run All is the workflow checkpoint | A result you only saw before a restart does not count |
| Repeated code becomes one function in `src/icd_project/` | If your group wrote the W06 reading function, use it in section 4: C1 must still pass |
| AI suggestions are verified before you accept them | Record each one in the AI use log of section 9 |
| A verified change is recorded in a small commit that says why | One commit when the notebook runs from start to end |

**Input:** `data/raw/teaching_publications_2020_2026_textual.csv` (the same file as in W06).
**Decision of the day:** the abstract is the text; the title identifies the document and is read
by people, but it does not enter the representation. The reference representation is the course's
configuration L3 (lowercase, English stop words, single words and pairs of words).
**Outputs:** `outputs/text_representation/` (CSV tables, one figure and `statistics.json`).

W07 starts again from the CSV file. It does not read anything W06 wrote in `outputs/`.

## 0 Setup and contract of the analysis

The first cell (above) found your project folder and defined `DATA_DIR` (`data/raw`) and
`OUTPUT_DIR` (`outputs`). The next cell imports the libraries and records their versions, because a
text representation depends on the version of the software that built it.

In [ ]:
import hashlib
import json
import math
import re
import textwrap
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 160)   # titles and text snippets are read, not skimmed
VERSIONS = {name: version(name) for name in ("pandas", "numpy", "matplotlib", "scikit-learn")}
VERSIONS

**Parameters.** One cell with every choice the representation depends on. Do not edit it during the
class, except `MY_CONTRAST` when section 8 tells you to: the checkpoints assume these values.

In [ ]:
PUBLICATIONS_FILE = "teaching_publications_2020_2026_textual.csv"   # one row per publication
TEXT_FIELD = "abstract"         # the text that enters the representation
TITLE_FIELD = "title"           # shown to people, never vectorised
SCENARIO = "textual"            # the W06 universe: all document types, 2020–2026, 2026 partial

# The reference TF-IDF settings, identical to the course's analytics (configuration L3).
TFIDF_REFERENCE = dict(
    analyzer="word", lowercase=True, strip_accents=None,
    token_pattern=r"(?u)\b\w\w+\b",  # a token is a run of two or more letters or digits
    stop_words="english",            # scikit-learn's built-in English list
    ngram_range=(1, 2),              # single words and pairs of adjacent words
    min_df=5,                        # a feature must occur in at least 5 abstracts
    max_df=0.85,                     # ... and in at most 85 % of them
    max_features=3000,               # keep the 3000 most frequent candidates
    norm="l2", use_idf=True, smooth_idf=True, sublinear_tf=True, binary=False, dtype=np.float64,
)
# The four reference configurations: each step changes ONE decision.
CONFIGURATIONS = {
    "L0": {"lowercase": False, "stop_words": None, "ngram_range": (1, 1)},
    "L1": {"lowercase": True, "stop_words": None, "ngram_range": (1, 1)},
    "L2": {"lowercase": True, "stop_words": "english", "ngram_range": (1, 1)},
    "L3": {"lowercase": True, "stop_words": "english", "ngram_range": (1, 2)},   # = reference
}
ALLOWED_CONTRASTS = [("L0", "L1"), ("L1", "L2"), ("L2", "L3")]
MY_CONTRAST = ("L2", "L3")      # section 8: replace with the contrast your group was given
CANDIDATE_KEYS = ("lowercase", "stop_words", "ngram_range", "token_pattern", "min_df", "max_df")

# Two pairs of publications fixed by the course for the practice (nearest neighbours in the
# reference representation). The slides read two other pairs; these two are yours.
FIXED_PAIRS = {
    "pair_1": ("2-s2.0-105027940742", "2-s2.0-85134800145"),
    "pair_2": ("2-s2.0-105005506378", "2-s2.0-105010831450"),
}
STRUCTURED_EXAMPLE = "2-s2.0-105024262520"  # an abstract with publisher headings, sections 4 and 6

**Reference values.** The course built the same representation once in its analytics and recorded
the results with the fingerprint (SHA-256 hash) of the input file. Your checkpoints compare with
these values. A hash that differs means your file is not the reference file: stop and tell the
teacher; do not adjust your results to match.

In [ ]:
REFERENCE = {
    "input_sha256": "60b1ab3f6ce49d7edcb65506f5adffb9c3f279f3ea1065e4240b3744a87961be",
    "n_documents": 3343,
    "matrix_shape": (3343, 3000),
    "matrix_nnz": 257842,
    "n_unigrams": 2468,
    "n_bigrams": 532,
    "candidate_vocabulary": {"L0": 6539, "L1": 5938, "L2": 5698, "L3": 11516},
    "retained_features": {"L0": 3000, "L1": 3000, "L2": 3000, "L3": 3000},
    "nnz": {"L0": 319383, "L1": 314336, "L2": 243154, "L3": 257842},
    "zero_vector_documents": {"L0": 1, "L1": 0, "L2": 0, "L3": 0},
    "df_aim_paper": 30,
    "cosine": {"pair_1": 0.287267, "pair_2": 0.232662},
    # Scale: cosine between abstracts of a fixed sample, 75 exclusive publications per group.
    "sample_cosine_median": {"within": 0.041152, "between": 0.033881},
    "sample_cosine_p90": {"within": 0.096275, "between": 0.074413},
}

**Three distinctions to keep in mind all session.**

- *Fictitious corpus vs. real corpus.* Sections 1–3 use four made-up sentences (D1–D4). Their
  numbers are **not** GOVCOPP results.
- *The text vs. its representation.* The matrix keeps counts of features, not sentences. Every
  interpretation goes back to the original abstract.
- *Reference configuration vs. your contrast.* L3 is the course's reference. Your contrast in
  section 8 is a second scenario, reported next to it; it never replaces it.

**From the slides to this notebook.** Each idea from the first part of the class becomes a few
lines of code here. Use the slide title to find the explanation again.

| Slides | Idea | Section | Object in the code |
|---|---|---|---|
| «Bag of words: from a sentence to a row of numbers» | tokens, vocabulary, document-feature matrix | 1 | `toy_counts`, `toy_df` |
| «TF-IDF in words, then in symbols»; «Worked example: four sentences, every arithmetic step» | idf, weights, rows of length 1 | 2 | `toy_idf`, `d4_manual` |
| «Cosine similarity: comparing directions, not sizes» | the cosine as a sum of products | 3, 7 | `toy_cosines`, `shared_terms` |
| «Corpus: the set of texts we decided to analyse»; «Read one abstract before deciding anything»; «And one that is not really an abstract» | corpus, textual unit, field rule, language check | 4 | `corpus`, `n_words`, `stopword_share` |
| «The preprocessing decision is what creates the feature»; «The corpus, formalised» | `min_df`, `max_df`, `max_features`; the reference matrix | 5 | `X`, `terms`, `candidates_reference` |
| «The same rule on a real abstract»; «Bigrams: new columns at the cost of old ones»; «A feature nobody ever wrote» | reading weights; features nobody wrote | 6 | `top_terms`, `doc_freq`, `in_context` |
| «Pair 1…», «Pair 2…»; «What counts as a high cosine here»; «Four reasons two abstracts look alike» | reading a cosine against the text; its scale | 7 | `pairs_table`, `pair_1_terms`, `pair_2_terms` |
| «Four configurations, one factor at a time»; «What lowercasing actually bought us»; «The stop-word list removes more than you asked»; «Representation sensitivity» | one decision changed, the rest fixed | 8, 10 | `CONFIGURATIONS`, `fitted`, `lost`, `gained`, `robustness` |
| «Sensitivity is only evidence if it is recorded» | checkpoints and the evidence record | 5, 7, 8, 9 | `REFERENCE`, `statistics` |

## 1 Fictitious corpus: from sentences to counts

*Fictitious data, written for this exercise. Not GOVCOPP data.* These are the four sentences of
the slides «Bag of words: from a sentence to a row of numbers» and «Worked example: four sentences,
every arithmetic step».

| Document | Text |
|---|---|
| D1 | Urban policy for cities |
| D2 | Urban transport policy |
| D3 | Public policy for cities |
| D4 | Public health policy |

**Before running anything:** on paper, cross out the word that carries no content ("for"), then
write one column per remaining word and one row per document, and count. How many columns do you
get?

In [ ]:
toy = pd.DataFrame({
    "doc": ["D1", "D2", "D3", "D4"],
    "text": ["Urban policy for cities", "Urban transport policy",
             "Public policy for cities", "Public health policy"],
}).set_index("doc")
N_TOY = len(toy)

toy_counter = CountVectorizer(stop_words="english")          # lowercase and English stop words, as in the reference
toy_counts = pd.DataFrame(toy_counter.fit_transform(toy["text"]).toarray(),
                          index=toy.index, columns=toy_counter.get_feature_names_out())
toy_counts

This table is a **document-term matrix**: one row per document, one column per term (a *feature*),
and in each cell the number of times the term occurs in the document. Word order is gone: "urban
policy" and "policy urban" would give the same row.

**Exercise 1.** Count, for every term, the number of documents in which it occurs at least once.
This is its **document frequency**.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError("Complete this cell")
toy_df

In [ ]:
# Checkpoint — a document frequency counts documents, not occurrences.
assert toy_df["policy"] == N_TOY, "'policy' occurs in every document."
assert toy_df["health"] == 1 and toy_df["urban"] == 2 and toy_df["public"] == 2
assert (toy_df >= 1).all() and (toy_df <= N_TOY).all()

## 2 From counts to TF-IDF

A raw count favours words that are frequent everywhere. **TF-IDF** (term frequency–inverse
document frequency) gives a term a high weight when it is **frequent in this document and rare in
the corpus**. "policy" occurs in all four documents, so it says nothing about any one of them;
"health" occurs only in D4, so it distinguishes D4.

**In symbols.** Write $N$ for the number of documents, $\mathrm{tf}_{dt}$ for the count of term $t$
in document $d$ and $\mathrm{df}_t$ for its document frequency. scikit-learn computes the inverse
document frequency as

$$\mathrm{idf}_t = \ln\frac{1 + N}{1 + \mathrm{df}_t} + 1$$

Read: "the natural logarithm of N plus one over the document frequency plus one, plus one". The
rarer the term, the larger the fraction and the larger the weight; the "+1" terms only stop a
division by zero and keep every weight above zero. The weight of $t$ in $d$ is then

$$w_{dt} = \mathrm{tf}_{dt} \times \mathrm{idf}_t$$

and each document's row is finally divided by its length $\lVert w_d \rVert = \sqrt{\sum_t
w_{dt}^2}$ (read: "the square root of the sum of the squared weights"), so that a long abstract
and a short one are comparable. In the fictitious corpus, $N = 4$; in GOVCOPP, $N = 3343$ abstracts.

**Exercise 2.** Compute the idf of every term from `toy_df` with the formula above (`np.log` is the
natural logarithm).

In [ ]:
# YOUR CODE HERE
raise NotImplementedError("Complete this cell")
toy_idf.round(4)

In [ ]:
# Checkpoint — your idf and scikit-learn's must agree.
toy_tfidf_vectorizer = TfidfVectorizer(stop_words="english", sublinear_tf=False)
toy_tfidf = pd.DataFrame(toy_tfidf_vectorizer.fit_transform(toy["text"]).toarray(),
                         index=toy.index, columns=toy_tfidf_vectorizer.get_feature_names_out())
assert np.allclose(toy_idf.to_numpy(), toy_tfidf_vectorizer.idf_), "Use N = 4 and the document frequency, not the counts."
assert math.isclose(toy_idf["policy"], 1.0), "A term in every document gets the smallest idf: ln(5/5) + 1 = 1."
assert toy_idf["health"] > toy_idf["public"] > toy_idf["policy"], "Rarer terms get a larger idf."

Now the whole row of D4, by hand, as on the slide «Worked example: four sentences, every
arithmetic step»: count × idf, then divide by the length of the row.

In [ ]:
d4_raw = toy_counts.loc["D4"] * toy_idf
d4_manual = d4_raw / np.sqrt((d4_raw ** 2).sum())
pd.DataFrame({"count": toy_counts.loc["D4"], "idf": toy_idf.round(4), "count × idf": d4_raw.round(4),
              "normalised (by hand)": d4_manual.round(4), "scikit-learn": toy_tfidf.loc["D4"].round(4)})

In [ ]:
# Checkpoint — the hand computation reproduces the library.
assert np.allclose(d4_manual, toy_tfidf.loc["D4"]), "Multiply count by idf, then divide by the square root of the sum of squares."
assert math.isclose((toy_tfidf ** 2).sum(axis=1).min(), 1.0), "After normalisation every row has length 1."

**Two more decisions in the GOVCOPP representation.** The reference adds `sublinear_tf=True`: a term
that occurs twice counts $1 + \ln 2 \approx 1.69$ instead of 2 (read: "one plus the logarithm of the
count"), so repetition helps less and less. And it adds `min_df`, `max_df` and `max_features`, which
decide which terms become columns at all (section 5).

**Your answer.** In D4, "health" has the largest weight (0.727) and "policy" the smallest (0.379).
Does that mean health matters more than policy in D4? What does the weight actually measure?

*Write here.*

## 3 Cosine similarity between documents

Each normalised row is a list of weights, one per term. Two documents are lexically close when
they give weight to the **same terms**. The **cosine similarity** multiplies the two rows term by
term and adds the products. Because every row has length 1, the result goes from 0 (no term in
common) to 1 (the same weights).

**In symbols.** For two documents $a$ and $b$ with rows $w_a$ and $w_b$:

$$\cos(a, b) = \frac{\sum_t w_{at}\, w_{bt}}{\lVert w_a \rVert\, \lVert w_b \rVert} = \sum_t w_{at}\, w_{bt} \quad \text{when both rows have length 1}$$

Read: "add, over all terms, the weight in a times the weight in b, and divide by the two lengths".
Only the terms present in **both** documents contribute: for any other term one of the two weights
is zero. That is why, in section 7, we can list *which* shared terms make up a cosine.

**Exercise 3.** Compute the cosine between D3 and D4 by hand: multiply their two rows of `toy_tfidf`
and add.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError("Complete this cell")
round(toy_cos_d3_d4, 4)

In [ ]:
# Checkpoint — the hand computation and the library agree.
toy_cosines = pd.DataFrame(cosine_similarity(toy_tfidf), index=toy.index, columns=toy.index)
assert math.isclose(toy_cos_d3_d4, toy_cosines.loc["D3", "D4"]), "Multiply term by term, then add: the rows already have length 1."
assert (toy_cosines > 0).all().all(), "Every pair shares 'policy', so no cosine is zero."
toy_cosines.round(4)

**Your answer.** D1–D2 and D3–D4 have exactly the same cosine (0.53). Are the two pairs equally
close in theme? What produces each cosine?

*Write here.*

## 4 The corpus and the textual unit (minutes 100–125)

**Provenance.** Scientific production of GOVCOPP researchers indexed in Scopus, 2020–2026,
extracted on 2026-09-16 (2026 is incomplete) and regenerated on 2026-10-10. This is the W06
`textual` universe: every publication in it has an abstract. DOIs and titles can identify authors:
never publish tables with individual researchers.

- `teaching_publications_2020_2026_textual.csv`: one row per **publication**, with its title and
  abstract.

The **corpus** is the set of documents we analyse; the **textual unit** is what one document is.
Here: one document = one publication = its abstract. The abstract is the authors' summary, not the
full text.

In [ ]:
publications = pd.read_csv(DATA_DIR / "teaching_publications_2020_2026_textual.csv", dtype=str, keep_default_na=False, na_values=[""])
corpus = publications.sort_values("publication_id", kind="stable").set_index("publication_id")
print("publications:", publications.shape)
corpus[[TITLE_FIELD, TEXT_FIELD, "govcopp_groups", "year"]].head(3)

In [ ]:
# Is this the reference file? Same bytes, same fingerprint.
input_sha256 = hashlib.sha256((DATA_DIR / PUBLICATIONS_FILE).read_bytes()).hexdigest()
assert input_sha256 == REFERENCE["input_sha256"], "Your data file differs from the reference file (re-saved in Excel?). Tell the teacher; do not adjust results."

**The field rule.** The abstract enters the representation; the title does not. The title is how
people recognise a document, and we print it next to every result. Adding it to the abstract would
be a different representation, with different numbers: if your group thinks it matters, record it in
the decision log (section 9) as an alternative, without computing it today.

In [ ]:
corpus_check = pd.Series({
    "documents (rows)": len(corpus),
    "publication_id repeated": int(corpus.index.duplicated().sum()),
    "abstract missing": int(corpus[TEXT_FIELD].isna().sum()),
    "abstract_available = yes": int((corpus["abstract_available"] == "yes").sum()),
    "title missing": int(corpus[TITLE_FIELD].isna().sum()),
}, name="value")
corpus_check

**Exercise 4.** Count the words of every abstract (split on spaces) and find the publication with the
shortest abstract.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError("Complete this cell")
n_words.describe().round(1)

In [ ]:
# Checkpoint — one abstract per publication, none of them empty.
assert len(n_words) == REFERENCE["n_documents"] and (n_words > 0).all(), "Every publication in the textual universe has an abstract."
assert corpus_check["publication_id repeated"] == 0 and corpus_check["abstract missing"] == 0
print(shortest_id, n_words[shortest_id], "words:", corpus.at[shortest_id, TITLE_FIELD])

**A language check.** The English stop-word list only makes sense for English text. A simple check
without new libraries: in an English abstract, a good share of the words are English function words
("the", "of", "and"). An abstract with a very low share deserves to be read.

In [ ]:
tokens = corpus[TEXT_FIELD].str.lower().str.findall(r"\b\w\w+\b")      # the same token rule as TF-IDF
stopword_share = tokens.map(lambda words: sum(w in ENGLISH_STOP_WORDS for w in words) / len(words))
print(stopword_share.describe().round(3).to_dict())
lowest = stopword_share.nsmallest(3).index
corpus.loc[lowest, [TITLE_FIELD]].assign(stopword_share=stopword_share[lowest].round(3),
                                         abstract_start=corpus.loc[lowest, TEXT_FIELD].str[:160])

**Your answers.**

1. Read the abstract with the lowest share. Is it an abstract? In which language(s)?
2. Should your group delete it from the corpus today? Who decided the universe, and where?

*Write here.*

**Two real abstracts before any processing.** First, an abstract written to a publisher's template:
it carries the headings *Purpose*, *Design/methodology/approach*, *Findings* and *Originality/value*.
About 263 abstracts of the corpus follow this template. Keep these headings in mind for section 6.

In [ ]:
def show(publication_id, width=100, limit=900):
    """Print the title, groups and the start of one abstract, wrapped for reading."""
    row = corpus.loc[publication_id]
    print(f"{publication_id} | {row['govcopp_groups']} | {row['year']}\nTITLE: {row[TITLE_FIELD]}")
    print(textwrap.fill(row[TEXT_FIELD][:limit] + (" […]" if len(row[TEXT_FIELD]) > limit else ""), width))

show(STRUCTURED_EXAMPLE)

**What the literature says — the corpus becomes counts.** The simplest representation of a text, the
*bag of words*, describes each document only by how often each word occurs in it. It discards word
order and most of the grammar, and in exchange it gives a compact table that many methods can use
(Grimmer, Roberts & Stewart (2022), *Text as Data*, “5.1 The Bag of Words Model”). Today's representation is a bag of words with weights: you have to know what it threw
away before you interpret it.

## 5 The reference representation — checkpoint C1 (minutes 100–125)

One line builds the course's representation from the abstracts: tokenise, lowercase, remove English
stop words, form single words and pairs of adjacent words, keep the candidates that occur in at
least 5 and at most 85 % of the abstracts, keep the 3000 most frequent of those, and weight them with
TF-IDF. The result is a **sparse** matrix: only the non-zero cells are stored.

**What the literature says — tokens and the document-feature matrix.** Tokenising cuts a document
into units (here, runs of two or more letters or digits); the distinct units across the corpus form
the vocabulary (Grimmer, Roberts & Stewart (2022), *Text as Data*, “5.3 Tokenize”). Counting each vocabulary unit in each document gives a matrix with one row
per document and one column per feature, and because each document uses only a small part of the
vocabulary, almost all of its cells are zero (Grimmer, Roberts & Stewart (2022), *Text as Data*, “5.5 Construct Document-Feature Matrix”).

In [ ]:
reference_vectorizer = TfidfVectorizer(**TFIDF_REFERENCE)
X = reference_vectorizer.fit_transform(corpus[TEXT_FIELD]).tocsr()
terms = reference_vectorizer.get_feature_names_out()
row_of = {pid: i for i, pid in enumerate(corpus.index)}       # publication_id → row of X
print(type(X).__name__, X.shape)

**In symbols — the reference weights.** With `sublinear_tf=True` the weight of term $t$ in abstract
$d$, before normalisation, is

$$w_{dt} = \left(1 + \ln \mathrm{tf}_{dt}\right) \times \mathrm{idf}_t \quad\text{for } \mathrm{tf}_{dt} > 0, \qquad w_{dt} = 0 \text{ otherwise}$$

and then every row is divided by its length, as in section 2. Read: "one plus the log of the count,
times the idf". A row of $X$ is one abstract; a column is one feature; $X$ has $N = 3343$ rows and
$J = 3000$ columns (read: "N documents by J features").

**Exercise 5.** From `X` and `terms`, compute the number of rows and columns, the **density** (the
share of cells that are not zero) and the number of features that are pairs of words (bigrams
contain a space).

In [ ]:
# YOUR CODE HERE
raise NotImplementedError("Complete this cell")
n_rows, n_features, round(density, 5), n_bigram_features

In [ ]:
# Checkpoint C1 — the representation is the reference one.
candidates_reference = CountVectorizer(**{k: TFIDF_REFERENCE[k] for k in CANDIDATE_KEYS}).fit(corpus[TEXT_FIELD])
assert (n_rows, n_features) == REFERENCE["matrix_shape"], "One row per abstract, one column per retained feature."
assert X.nnz == REFERENCE["matrix_nnz"], "Check the parameters cell: the matrix differs from the reference."
assert n_bigram_features == REFERENCE["n_bigrams"] and n_features - n_bigram_features == REFERENCE["n_unigrams"]
assert len(candidates_reference.vocabulary_) == REFERENCE["candidate_vocabulary"]["L3"]
assert np.allclose(np.sqrt(X.multiply(X).sum(axis=1)), 1.0), "Every abstract row has length 1 after normalisation."

**Checkpoint C1 — record it before you go on.**

| Quantity | Your value |
|---|---|
| documents (rows) | |
| candidate vocabulary (after `min_df` and `max_df`) | |
| retained features (columns) | |
| single words / pairs of words | |
| density | |
| abstracts with no retained feature | |

## 6 Reading the representation

A row of `X` can be read back as a list of weighted terms. We do it for one abstract, and then we
look at the pairs of words.

In [ ]:
doc_freq = pd.Series(np.asarray((X > 0).sum(axis=0)).ravel(), index=terms, name="document_frequency")

def top_terms(publication_id, k=10):
    """The k terms with the largest TF-IDF weight in one abstract, with their document frequency."""
    row = X[row_of[publication_id]].toarray().ravel()
    order = np.argsort(-row, kind="stable")[:k]
    return pd.DataFrame({"term": terms[order], "tfidf": row[order].round(3),
                         "document_frequency": doc_freq.iloc[order].to_numpy()})

top_terms(STRUCTURED_EXAMPLE, k=12)

The slides read the Genova abstract this way. Here, the sparkling-wine abstract of section 4. Where
are the words that say what the paper is about?

In [ ]:
weights = pd.Series(X[row_of[STRUCTURED_EXAMPLE]].toarray().ravel(), index=terms)
rank_of = weights.rank(ascending=False, method="first")
pd.DataFrame([{"word": w, "is a feature": w in doc_freq.index,
               "abstracts containing it": int(corpus[TEXT_FIELD].str.lower().str.contains(rf"\b{w}\b", regex=True).sum()),
               "weight here": round(float(weights.get(w, 0.0)), 3),
               "rank in this abstract": int(rank_of[w]) if w in doc_freq.index and weights[w] > 0 else None}
              for w in ("sparkling", "winemaking", "fermentation", "bottling", "wine", "value research")])

**Your answer.** The paper is about sparkling winemaking. Which of its own words are not features
at all, and which setting removed them? What is "value research", and why is it among the heaviest
features of this abstract?

*Write here.*

**Pairs of words.** Bigrams can keep phrases that single words break apart, such as "climate
change". But they are formed **after** stop words are removed, and a token stops at any character
that is not a letter or a digit.

In [ ]:
bigram_df = doc_freq[[" " in t for t in doc_freq.index]].sort_values(ascending=False)
bigram_df.head(20).to_frame()

**Exercise 6.** The feature "aim paper" is a bigram of the reference representation. Find its
document frequency in `doc_freq`, and count the abstracts whose lowercased text contains the literal
string "aim paper".

In [ ]:
# YOUR CODE HERE
raise NotImplementedError("Complete this cell")
feature_df_aim_paper, literal_aim_paper

In [ ]:
# Checkpoint — a bigram feature is not a phrase of the text.
assert feature_df_aim_paper == REFERENCE["df_aim_paper"], "Look the term up in doc_freq: its index is the feature name."
assert literal_aim_paper <= feature_df_aim_paper, "Every literal 'aim paper' also produces the feature."

A helper to see a feature in its original context. It looks for the words of a feature as written,
allowing a hyphen or punctuation between them, and prints a few words around each hit.

In [ ]:
def in_context(term, n=3, width=60):
    """Up to n abstracts where the words of `term` occur next to each other, with surrounding text."""
    flags = 0 if any(ch.isupper() for ch in term) else re.IGNORECASE
    pattern = re.compile(r"\b" + r"\W+".join(map(re.escape, term.split())) + r"\b", flags)
    hits = []
    for pid, text in corpus[TEXT_FIELD].items():
        match = pattern.search(text)
        if match:
            start, end = max(0, match.start() - width), match.end() + width
            hits.append({"publication_id": pid, "context": "…" + text[start:end].replace("\n", " ") + "…"})
        if len(hits) == n:
            break
    if not hits:
        print(f"'{term}': no abstract has these words next to each other; the feature was built after removing stop words.")
    return pd.DataFrame(hits)

in_context("covid 19")

In [ ]:
in_context("aim paper")

**Your answers.**

1. Where does the feature "aim paper" come from, given that no abstract says "aim paper"?
2. Among the 20 most frequent bigrams, find two that describe the *format* of an abstract and two
   that describe its *content*.

*Write here.*

## 7 Two fixed pairs — checkpoint C2 (minutes 125–150)

The course searched, for a few publications, the abstracts with the highest cosine in the reference
representation. The slides read two of those neighbour pairs with you: Genova ↔ Tampere (the
proximity holds in the text) and the diversity-and-inclusion book ↔ the gamification handbook (the
proximity comes from the publisher's blurb). Two other pairs are fixed for the practice, and nobody
has read them for you. Both join a publication of SAD to one of CIS. Each group reads both: the
number first, then the shared terms that produce it, then the two abstracts.

**The scale of a cosine.** Between two abstracts of the same formal group, the median cosine is
about 0.04; between groups, about 0.03; nine pairs in ten stay below 0.10 (a fixed sample of 75
exclusive publications per group). A cosine of 0.2 or 0.4 is therefore very high *for this corpus*.

**What the literature says — the cosine.** A document can be treated as a vector of term weights,
one dimension per vocabulary term, and two documents compared through the angle between their
vectors (Menczer, Fortunato & Davis (2020), *A First Course in Network Science*, “Box 4.1 Cosine Similarity”). Because the cosine divides by the lengths of the two vectors, a long and a short
document with the same proportions of words count as equally similar (Grimmer, Roberts & Stewart (2022), *Text as Data*, “7.1 Similarity Metrics”). Weighting by TF-IDF
before comparing makes the comparison depend more on distinctive words and less on words every
document uses (Grimmer, Roberts & Stewart (2022), *Text as Data*, “7.3 tf-idf Weighting”).

In [ ]:
pairs_table = pd.DataFrame([
    {"pair": name, "publication_a": a, "title_a": corpus.at[a, TITLE_FIELD], "groups_a": corpus.at[a, "govcopp_groups"],
     "publication_b": b, "title_b": corpus.at[b, TITLE_FIELD], "groups_b": corpus.at[b, "govcopp_groups"]}
    for name, (a, b) in FIXED_PAIRS.items()
]).set_index("pair")
pairs_table

**Exercise 7.** Compute the cosine of pair 1 directly from `X`: multiply the two rows cell by cell
and add (`.multiply(...)` then `.sum()`; the rows already have length 1).

In [ ]:
a, b = FIXED_PAIRS["pair_1"]
# YOUR CODE HERE
raise NotImplementedError("Complete this cell")
round(cosine_pair_1, 6)

In [ ]:
# Checkpoint C2 — the number matches the reference and the library.
assert math.isclose(cosine_pair_1, REFERENCE["cosine"]["pair_1"], abs_tol=1e-6), "Use the two rows of the reference matrix X, found with row_of."
assert math.isclose(cosine_pair_1, cosine_similarity(X[row_of[a]], X[row_of[b]])[0, 0])

**What makes the cosine.** Only the terms present in both abstracts contribute. The helper lists
them with their weights and the share of the cosine each one explains.

In [ ]:
def shared_terms(publication_a, publication_b, matrix=None, vocabulary=None, positions=None):
    """Terms present in both abstracts, with weights, product and share of the cosine."""
    matrix = X if matrix is None else matrix
    vocabulary = terms if vocabulary is None else vocabulary
    positions = row_of if positions is None else positions
    wa = matrix[positions[publication_a]].toarray().ravel()
    wb = matrix[positions[publication_b]].toarray().ravel()
    both = np.flatnonzero((wa > 0) & (wb > 0))
    table = pd.DataFrame({"term": vocabulary[both], "weight_a": wa[both], "weight_b": wb[both],
                          "product": wa[both] * wb[both]})
    table["share_of_cosine"] = table["product"] / table["product"].sum()
    return table.sort_values("product", ascending=False, ignore_index=True)   # round only to display

pair_1_terms = shared_terms(*FIXED_PAIRS["pair_1"])
print(len(pair_1_terms), "shared terms; the first five explain", round(pair_1_terms["share_of_cosine"].head(5).sum(), 3), "of the cosine")
pair_1_terms.head(12).round(4)

In [ ]:
show(FIXED_PAIRS["pair_1"][0], limit=900)
print()
show(FIXED_PAIRS["pair_1"][1], limit=900)

**Pair 2, the same steps.**

In [ ]:
cosine_pair_2 = float(X[row_of[FIXED_PAIRS["pair_2"][0]]].multiply(X[row_of[FIXED_PAIRS["pair_2"][1]]]).sum())
pair_2_terms = shared_terms(*FIXED_PAIRS["pair_2"])
print(round(cosine_pair_2, 6), "|", len(pair_2_terms), "shared terms; the first five explain",
      round(pair_2_terms["share_of_cosine"].head(5).sum(), 3), "of the cosine")
pair_2_terms.head(12).round(4)

In [ ]:
# Checkpoint C2 (second pair).
assert math.isclose(cosine_pair_2, REFERENCE["cosine"]["pair_2"], abs_tol=1e-6)
assert math.isclose(pair_2_terms["product"].sum(), cosine_pair_2, abs_tol=1e-9), "The shared terms add up to the cosine."

In [ ]:
show(FIXED_PAIRS["pair_2"][0], limit=900)
print()
show(FIXED_PAIRS["pair_2"][1], limit=900)

**The two pairs side by side.** For each pair, the eight shared terms that contribute most.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, (name, table, cosine) in zip(axes, [("pair 1", pair_1_terms, cosine_pair_1), ("pair 2", pair_2_terms, cosine_pair_2)]):
    top = table.head(8).iloc[::-1]
    ax.barh(top["term"], top["share_of_cosine"], color="#4D7EA8")
    ax.set_xlabel("share of the cosine")
    ax.set_title(f"{name}: cosine {cosine:.3f}, {len(table)} shared terms")
fig.suptitle("What each cosine is made of (reference representation L3, abstracts only)")
fig.tight_layout()
(OUTPUT_DIR / "text_representation").mkdir(exist_ok=True)
fig.savefig(OUTPUT_DIR / "text_representation" / "fixed_pairs_shared_terms.png", dpi=150, bbox_inches="tight")
plt.show()

**Checkpoint C2 — your reading.** For each pair, one sentence: is the lexical proximity supported by
the two abstracts? Quote one phrase from the original text that supports or contradicts it, and name
one thing the cosine does not show.

| Pair | Cosine | Supported by the text? | Phrase from the abstracts | What the cosine does not show |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |

## 8 One contrast, one factor — checkpoint C3 (minutes 150–180)

The course compared four configurations. Each step changes **one** decision and keeps all the
others, so any difference between two neighbours has one cause.

| Configuration | Lowercase | Stop words | n-grams | Candidates | Retained | Non-zero cells | Abstracts with no feature |
|---|---|---|---|---:|---:|---:|---:|
| L0 | no | none | single words | 6539 | 3000 | 319383 | 1 |
| L1 | yes | none | single words | 5938 | 3000 | 314336 | 0 |
| L2 | yes | English list | single words | 5698 | 3000 | 243154 | 0 |
| L3 (reference) | yes | English list | single words and pairs | 11516 | 3000 | 257842 | 0 |

Your group investigates **one** contrast, the one the teacher gave you: L0 → L1 (lowercase),
L1 → L2 (stop words) or L2 → L3 (bigrams). This is not a search for the best configuration.

**Set `MY_CONTRAST` in the parameters cell (section 0) now, then run Restart & Run All.**

**What the literature says — preprocessing is a decision.** Before counting, analysts usually drop
part of the vocabulary to make it manageable, and each such step involves choices that change what
the representation contains (Grimmer, Roberts & Stewart (2022), *Text as Data*, “5.4 Reduce Complexity”). Much of that reduction serves readability or computation
rather than the question: a stop-word list makes word lists and figures cleaner, but the same effect
can be obtained by filtering the display instead of the input (Grimmer, Roberts & Stewart (2022), *Text as Data*, “5.6.2 THE SCALE ARGUMENT AGAINST PREPROCESSING”).

**Exercise 8.** List the settings whose value differs between the two configurations of your
contrast (compare `CONFIGURATIONS[config_a]` and `CONFIGURATIONS[config_b]` key by key).

In [ ]:
config_a, config_b = MY_CONTRAST
# YOUR CODE HERE
raise NotImplementedError("Complete this cell")
changed

In [ ]:
# Checkpoint — a contrast changes exactly one decision.
assert tuple(MY_CONTRAST) in ALLOWED_CONTRASTS, "Use one of the three contrasts between neighbouring configurations."
assert len(changed) == 1, "A one-factor contrast changes exactly one setting; the others stay as in the reference."

In [ ]:
def fit_configuration(name):
    """TF-IDF with the reference settings and the decisions of configuration `name`."""
    params = {**TFIDF_REFERENCE, **CONFIGURATIONS[name]}
    vectorizer = TfidfVectorizer(**params)
    matrix = vectorizer.fit_transform(corpus[TEXT_FIELD]).tocsr()
    candidates = CountVectorizer(**{k: params[k] for k in CANDIDATE_KEYS}).fit(corpus[TEXT_FIELD])
    vocabulary = vectorizer.get_feature_names_out()
    return {"X": matrix, "terms": vocabulary, "candidate_vocabulary": len(candidates.vocabulary_),
            "document_frequency": pd.Series(np.asarray((matrix > 0).sum(axis=0)).ravel(), index=vocabulary)}

fitted = {name: fit_configuration(name) for name in MY_CONTRAST}
contrast_summary = pd.DataFrame({
    name: {"candidate vocabulary": m["candidate_vocabulary"], "retained features": m["X"].shape[1],
           "non-zero cells": m["X"].nnz, "abstracts with no feature": int((m["X"].getnnz(axis=1) == 0).sum())}
    for name, m in fitted.items()
})
contrast_summary

In [ ]:
# Checkpoint C3 — both configurations are the course's.
for name, m in fitted.items():
    assert m["candidate_vocabulary"] == REFERENCE["candidate_vocabulary"][name], f"{name}: check that only CONFIGURATIONS[{name!r}] overrides the reference settings."
    assert m["X"].nnz == REFERENCE["nnz"][name] and m["X"].shape[1] == REFERENCE["retained_features"][name]
    assert int((m["X"].getnnz(axis=1) == 0).sum()) == REFERENCE["zero_vector_documents"][name]

**Exercise 9.** Which features does your contrast lose and which does it gain? Compare the two
vocabularies as sets: `lost` holds the terms of `config_a` that are not in `config_b`, `gained` the
opposite. Sort both.

In [ ]:
terms_a, terms_b = fitted[config_a]["terms"], fitted[config_b]["terms"]
# YOUR CODE HERE
raise NotImplementedError("Complete this cell")
len(lost), len(gained)

In [ ]:
# Checkpoint — with 3000 retained features on both sides, every feature lost frees a place for one gained.
kept = set(terms_a) & set(terms_b)
assert len(kept) + len(lost) == len(terms_a) and len(kept) + len(gained) == len(terms_b), "Lost = in a, not in b; gained = in b, not in a."
assert len(lost) == len(gained)

In [ ]:
lost_table = fitted[config_a]["document_frequency"][lost].sort_values(ascending=False).rename("document_frequency")
gained_table = fitted[config_b]["document_frequency"][gained].sort_values(ascending=False).rename("document_frequency")
print(f"{config_a} → {config_b}: {len(kept)} features kept, {len(lost)} lost, {len(gained)} gained")
pd.concat({f"lost (in {config_a})": lost_table.head(15).rename_axis("term").reset_index(),
           f"gained (in {config_b})": gained_table.head(15).rename_axis("term").reset_index()}, axis=1)

**Back to the text.** Pick one lost and one gained feature that matter for reading GOVCOPP
abstracts and that the slides did not discuss (not `The`/`This`, not `system`/`de`, not `aim paper`),
and look at them in context with `in_context("…")`. For the stop-word contrast, look also at the
words of the English list that occur in your corpus.

In [ ]:
in_context(gained_table.index[0])

In [ ]:
if (config_a, config_b) == ("L1", "L2"):
    # The words of the English list that were features before the list was applied.
    stopwords_in_corpus = fitted[config_a]["document_frequency"].reindex(sorted(ENGLISH_STOP_WORDS)).dropna()
    display(stopwords_in_corpus.astype(int).sort_values(ascending=False).to_frame("document_frequency").T)
else:
    print("This cell is for the L1 → L2 contrast only.")

**Checkpoint C3 — your reading.** Two to four sentences: what does your contrast preserve, what does
it remove, and does the change help or hurt the project question? Quote one real abstract.

*Write here.*

**Where today's work goes in the project.**

- **Common trunk (phase 3: textual representation and similarity).** Sections 4–8 are your evidence
  for this phase and close **M2** (deadline 3 November 2026, 23:59): M2 stops at the representation.
  No topics, clusters or classifiers enter M2.
- **Evidence entry E07** in `docs/evidence_register.md`: the corpus and unit (publication, abstract),
  the field rule, the preprocessing and n-gram settings, the matrix dimensions, your one-factor
  contrast, the two pairs and their limits, and any AI use.
- **W08** rebuilds TF-IDF from the CSV file with the settings you documented today; it does not read
  today's outputs. A setting you did not record cannot be rebuilt.
- **The project question.** A cosine between two abstracts is evidence of shared vocabulary. Whether
  shared vocabulary follows the formal groups is a question for W08–W09, and it needs the reading
  habit of section 7.

## 9 Decision log, evidence record and outputs (minutes 150–180)

**Decision log.** In the *Reason* column, connect each choice to the criterion quoted in this
notebook and to your own result (a term, a pair, a number).

| Decision | Alternative considered | Reason | Effect on the result |
|---|---|---|---|
| Text = abstract; title only to identify | title + abstract | | |
| Stop words = scikit-learn English list | no list (L1); a list adapted to this corpus | | |
| Single words and pairs of words (L3) | single words only (L2) | | |
| `min_df=5`, `max_df=0.85`, `max_features=3000` | fixed by the course; not explored today | | |
| Our contrast: | | | |

**AI use log** (course policy).

| Tool and date | Request | Change made | How it was checked |
|---|---|---|---|
| | | | |

**What the literature says — reporting a sensitivity check.** A preprocessing step can change the
vocabulary in ways that matter for later results, so the decision belongs in the record together
with what it changed (Grimmer, Roberts & Stewart (2022), *Text as Data*, “5.6.2 THE SCALE ARGUMENT AGAINST PREPROCESSING”). Here the record is the parameters, the contrast table, the lost and
gained terms, and the hash of the input file.

**Save the outputs.** The tables contain publication IDs and titles, never researchers.

In [ ]:
(OUTPUT_DIR / "text_representation").mkdir(exist_ok=True)
configurations_table = pd.DataFrame({
    name: {**{k: str(v) for k, v in CONFIGURATIONS[name].items()},
           "candidate_vocabulary": REFERENCE["candidate_vocabulary"][name],
           "retained_features": REFERENCE["retained_features"][name], "nnz": REFERENCE["nnz"][name],
           "zero_vector_documents": REFERENCE["zero_vector_documents"][name],
           "recomputed_here": name in fitted or name == "L3"}
    for name in CONFIGURATIONS
}).T.rename_axis("configuration")
configurations_table.to_csv(OUTPUT_DIR / "text_representation" / "configuration_comparison.csv")
fixed_pairs = pairs_table.assign(
    cosine_L3=[cosine_pair_1, cosine_pair_2],
    top_shared_terms=["; ".join(t["term"].head(5)) for t in (pair_1_terms, pair_2_terms)],
    shared_terms_n=[len(pair_1_terms), len(pair_2_terms)],
)
fixed_pairs.to_csv(OUTPUT_DIR / "text_representation" / "fixed_pairs.csv")
contrast_terms = pd.concat([lost_table.rename_axis("term").reset_index().assign(change="lost", configuration=config_a),
                            gained_table.rename_axis("term").reset_index().assign(change="gained", configuration=config_b)])
contrast_terms.to_csv(OUTPUT_DIR / "text_representation" / "contrast_terms.csv", index=False)
statistics = {
    "input_file": PUBLICATIONS_FILE, "input_sha256": input_sha256, "scenario": SCENARIO,
    "text_field": TEXT_FIELD, "title_field_vectorised": False,
    "n_documents": int(n_rows), "matrix_shape": [int(n_rows), int(n_features)], "matrix_nnz": int(X.nnz),
    "density": density, "n_bigram_features": int(n_bigram_features),
    "candidate_vocabulary_L3": len(candidates_reference.vocabulary_),
    "abstract_words": {k: float(v) for k, v in n_words.describe().round(1).items()},
    "lowest_stopword_share": {"publication_id": stopword_share.idxmin(), "share": round(float(stopword_share.min()), 3)},
    "cosine_fixed_pairs_L3": {"pair_1": cosine_pair_1, "pair_2": cosine_pair_2},
    "contrast": {"from": config_a, "to": config_b, "changed": changed, "kept": len(kept),
                 "lost": len(lost), "gained": len(gained)},
    "parameters": {"tfidf_reference": {k: (list(v) if isinstance(v, tuple) else v) for k, v in TFIDF_REFERENCE.items() if k != "dtype"},
                   "configurations": {k: {kk: (list(vv) if isinstance(vv, tuple) else vv) for kk, vv in v.items()}
                                      for k, v in CONFIGURATIONS.items()},
                   "fixed_pairs": FIXED_PAIRS},
    "dependency_versions": VERSIONS,
}
(OUTPUT_DIR / "text_representation" / "statistics.json").write_text(json.dumps(statistics, indent=2), encoding="utf-8")
sorted(p.name for p in (OUTPUT_DIR / "text_representation").iterdir())

**What the group keeps from today.** (1) This notebook, after **Restart & Run All**, with the C1
table, the C2 and C3 readings and the decision log filled in. (2) The folder
`outputs/text_representation/` with the three CSV tables, the figure and `statistics.json`. (3) A
commit whose message says what you ran and which contrast you investigated. Nothing is handed in at
the end of the class: today's evidence enters the M2 dossier.

## 10 Optional extension: the fixed pairs under your contrast

*Only if you have finished.* Do the two fixed pairs keep their cosine when the representation
changes? This is the shape of a **robustness check**: same documents, one declared decision changed,
both results reported side by side.

**Exercise 10.** Compute the cosine of both fixed pairs in `fitted[config_a]` and `fitted[config_b]`.
Use `shared_terms(...)` with the `matrix` and `vocabulary` arguments, and sum the `product` column.

In [ ]:
robustness = {}
for name, m in fitted.items():
    for pair, (a, b) in FIXED_PAIRS.items():
        # YOUR CODE HERE
        raise NotImplementedError("Complete this cell")
robustness = pd.Series(robustness).unstack().round(4)
robustness

In [ ]:
# Checkpoint — a cosine stays between 0 and 1, and L3 gives back the reference.
assert ((robustness >= 0) & (robustness <= 1)).all().all()
if "L3" in robustness.index:
    assert math.isclose(robustness.loc["L3", "pair_1"], REFERENCE["cosine"]["pair_1"], abs_tol=1e-3)

**Your answer.** Did your contrast change the cosine of the two pairs? For the pair whose shared
vocabulary is the method (pair 2), can a preprocessing decision fix the misleading similarity?

*Write here.*

## After class — home extension (about 4 h per student)

You **apply** what was shown today. You do not add another representation, new libraries or a
search over settings, and you do not start topics, clustering or classification: M2 stops here.

1. **The second pair and the justification**, if they were not finished in class: complete the C2
   table and the C3 reading, including what the cosine does not show.
2. **Evidence entry E07** in `docs/evidence_register.md` (see the box at the end of section 8), and
   the decision log copied to `docs/analysis_decisions.md`.
3. **Research Plan v1** in `docs/research_plan_v1.md`: revise v0 after the W06 audit and today's
   representation. A justified choice of variant, two or three sub-questions at most, and limits.
4. **The M2 dossier** (deadline 3 November 2026, 23:59), following the M2 brief: E06 and E07, the
   W06 synthesis, the notebook versions, the data manifest and `docs/ai_use_declaration.md`.
5. **Individual run.** Each member runs this notebook once with Restart Kernel → Run All and records
   it in `docs/contributions.csv`.
6. **Optional, W05 «From repeated code to one function».** If your group moved the W06 reading rule
   into `src/icd_project/`, read the publications table with it here and run Restart Kernel → Run
   All: checkpoint C1 must pass unchanged.

Finish with a commit whose message says what changed and why.